# 03 — Avaliação do Pipeline RAG

Este notebook avalia o pipeline implementado em `02_RAG.ipynb`, reutilizando:
- Embeddings `models/gemini-embedding-001`;
- Coleção ChromaDB `olist_customer_reviews`;
- LLM `gemini-3.5-flash-lite`;
- Recuperação semântica Top-K e filtros por metadados.

**Importante:** a avaliação de recuperação exige um pequeno conjunto de referência (*ground truth*): para cada pergunta, alguém precisa identificar quais `review_id` realmente respondem à pergunta. Sem essa referência, não é possível calcular Recall@K ou Precision@K de forma confiável.

O notebook não inventa rótulos nem resultados. As métricas só serão calculadas para perguntas cujos `review_id` esperados forem preenchidos.

## 1. Preparar ambiente e reconectar à coleção existente

Execute este notebook na pasta `fase_4/notebooks` ou ajuste `project_root` para a raiz do repositório. O código abre a coleção persistida; não gera embeddings novamente nem adiciona documentos.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv, find_dotenv

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Ajuste caso o notebook não esteja em fase_4/notebooks
notebook_dir = Path.cwd().resolve()
project_root = notebook_dir.parents[1] if notebook_dir.name == "notebooks" else notebook_dir.parent

load_dotenv(find_dotenv())

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY não encontrada. Verifique o arquivo .env.")

diretorio_chroma = project_root / "fase_4" / "models" / "chroma_db"

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY
)

vectorstore = Chroma(
    collection_name="olist_customer_reviews",
    embedding_function=embeddings,
    persist_directory=str(diretorio_chroma)
)

print("Coleção:", "olist_customer_reviews")
print("Documentos na coleção:", vectorstore._collection.count())
print("Diretório:", diretorio_chroma)

Coleção: olist_customer_reviews
Documentos na coleção: 300
Diretório: /Users/ceciliafa/Documents/FIAP_1IAST/TechChallenges/fiap_tech_challenge_04/fiap_tech_challenge_04/fase_4/models/chroma_db


### Atenção ao tamanho da base

No `02_RAG.ipynb`, a criação dos documentos usa `df_clean.sample(n=min(300, len(df_clean)), random_state=42)`. Isso significa que aquela execução prepara até 300 documentos para indexação. Como o ChromaDB é persistente, confira a contagem impressa acima: execuções anteriores podem ter deixado documentos adicionais na coleção.

Para que a avaliação represente a base completa, será necessário indexar a base pretendida e registrar esse escopo no relatório.

## 2. Configurar o mesmo prompt e modelo de geração do projeto

In [2]:
template_rag = """Você é um especialista em Inteligência de Voz do Cliente (Voice of Customer Intelligence) analisando avaliações do e-commerce Olist.
Responda à pergunta do gestor baseando-se ESTRITAMENTE nas avaliações fornecidas.

Diretrizes:
1. Use apenas as evidências do contexto. Não invente fatos.
2. Cite os IDs das avaliações no formato [Review ID: ...].
3. Se o contexto não contiver informações suficientes ou a pergunta for fora de escopo, responda:
"Não foram encontradas avaliações suficientes na base de conhecimento sobre este assunto."

Contexto:
{context}

Pergunta:
{question}

Resposta Executiva Fundamentada:"""

prompt = ChatPromptTemplate.from_template(template_rag)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.1,
    google_api_key=GOOGLE_API_KEY
)

def formatar_documentos(docs):
    partes = []
    for doc in docs:
        md = doc.metadata
        partes.append(
            f"[Review ID: {md.get('review_id', 'N/A')} | "
            f"Nota: {md.get('review_score', 'N/A')} estrelas | "
            f"Categoria: {md.get('product_category_name', 'N/A')} | "
            f"Atraso: {md.get('entregue_com_atraso', 'N/A')} | "
            f"Dias de atraso: {md.get('dias_atraso', 'N/A')}]\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(partes)

def executar_rag_com_evidencias(pergunta, k=5, filtro=None):
    """Retorna resposta, documentos recuperados e distâncias do ChromaDB."""
    resultados = vectorstore.similarity_search_with_score(
        pergunta,
        k=k,
        filter=filtro
    )
    docs = [doc for doc, distancia in resultados]
    distancias = [float(distancia) for doc, distancia in resultados]

    contexto = formatar_documentos(docs)
    chain = prompt | llm | StrOutputParser()
    resposta = chain.invoke({
        "context": contexto,
        "question": pergunta
    })

    return {
        "resposta": resposta,
        "documentos": docs,
        "distancias": distancias
    }

## 3. Montar o conjunto de perguntas de avaliação

Comece com 20 a 30 perguntas variadas. Para cada pergunta respondível, preencha `review_ids_relevantes` com IDs que foram conferidos manualmente e que realmente sustentam uma resposta. Não use simplesmente os IDs que o próprio RAG retornou como verdade de referência.

- `filtro`: opcional; use a mesma sintaxe de filtros do ChromaDB.
- `espera_abstencao=True`: para perguntas sem evidências esperadas ou fora do escopo.
- Para perguntas respondíveis, preencha a lista de IDs relevantes.

Os exemplos abaixo são um ponto de partida: substitua os IDs vazios por IDs validados.

In [3]:
casos_avaliacao = [
    {
        "id_pergunta": "Q01",
        "pergunta": "Quais são os principais problemas relatados pelos clientes sobre entrega?",
        "review_ids_relevantes": ["95e7f613e3dfe80a79306a8c80084739", "65e3d04aef73660b0b2145ef3d5762cd", "a83253201bb3ee4c020d9b42f1cfac38", "e8d1bc0188ba7974e847024cab472cdb"],  # preencher após validação humana
        "filtro": None,
        "espera_abstencao": False
    },
    {
        "id_pergunta": "Q02",
        "pergunta": "O que os consumidores mais elogiam na experiência de compra?",
        "review_ids_relevantes": ["029f6693f6fa41f1acd19f729e4e8b39", "bf17681b8e58780d5064140aafa8f74d", "87bf7ea0b6a55642abc6b4dae8364d16", "87bf7ea0b6a55642abc6b4dae8364d16", "87bf7ea0b6a55642abc6b4dae8364d16", "7b89e676634561673670ec8b54ae202a", "7b89e676634561673670ec8b54ae202a"],  # preencher após validação humana
        "filtro": {"review_score": {"$gte": 4}},
        "espera_abstencao": False
    },
    {
        "id_pergunta": "Q03",
        "pergunta": "Quais problemas aparecem nas avaliações de clientes insatisfeitos?",
        "review_ids_relevantes": ["87fb462b15995bf3c5b6fba9d9678e04", "cb30c1d49c50132378faa2a855dbbb6d", "8d0f11dad4a5706b1a970995c6585605", "8d0f11dad4a5706b1a970995c6585605"],  # preencher após validação humana
        "filtro": {"review_score": {"$lte": 2}},
        "espera_abstencao": False
    },
    {
        "id_pergunta": "Q04",
        "pergunta": "Quais transtornos são relatados em pedidos com atraso?",
        "review_ids_relevantes": ["b7fe825a931a313b8aa2d18e8004b328", "d5b353235f98e304b065c6b74c57fd3d", "8d0f11dad4a5706b1a970995c6585605"],  # preencher após validação humana
        "filtro": {"entregue_com_atraso": True},
        "espera_abstencao": False
    },
    {
        "id_pergunta": "Q05",
        "pergunta": "Qual será a previsão do tempo amanhã em São Paulo?",
        "review_ids_relevantes": [],
        "filtro": None,
        "espera_abstencao": True
    },
]

# Confira se os IDs de referência estão preenchidos antes de calcular métricas de recuperação.
pd.DataFrame([
    {
        "id_pergunta": c["id_pergunta"],
        "pergunta": c["pergunta"],
        "qtd_ids_relevantes": len(c["review_ids_relevantes"]),
        "espera_abstencao": c["espera_abstencao"]
    }
    for c in casos_avaliacao
])

,id_pergunta,pergunta,qtd_ids_relevantes,espera_abstencao
0,Q01,Quais são os principais problemas relatados pe...,4,False
1,Q02,O que os consumidores mais elogiam na experiên...,7,False
2,Q03,Quais problemas aparecem nas avaliações de cli...,4,False
3,Q04,Quais transtornos são relatados em pedidos com...,3,False
4,Q05,Qual será a previsão do tempo amanhã em São Pa...,0,True


## 4. Executar as consultas e guardar as evidências

O código salva os IDs e textos recuperados, a distância devolvida pelo ChromaDB e a resposta do Gemini. **Distância não é percentual de acerto**: não interprete diretamente esse valor como precisão ou relevância percentual.

In [4]:
resultados_avaliacao = []

for caso in casos_avaliacao:
    resultado = executar_rag_com_evidencias(
        pergunta=caso["pergunta"],
        k=5,
        filtro=caso["filtro"]
    )

    docs = resultado["documentos"]
    ids_recuperados = [
        str(doc.metadata.get("review_id", ""))
        for doc in docs
    ]

    resultados_avaliacao.append({
        **caso,
        "ids_recuperados": ids_recuperados,
        "distancias": resultado["distancias"],
        "documentos": docs,
        "resposta": resultado["resposta"]
    })

    print(f"\n{'='*90}\n{caso['id_pergunta']} — {caso['pergunta']}")
    print("\nIDs recuperados:", ids_recuperados)
    print("\nRESPOSTA:\n", resultado["resposta"])
    for pos, doc in enumerate(docs, start=1):
        print(
            f"\nEvidência {pos} | Review ID: {doc.metadata.get('review_id')} "
            f"| Nota: {doc.metadata.get('review_score')} "
            f"| Atraso: {doc.metadata.get('entregue_com_atraso')}"
        )
        print(doc.page_content)

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Q01 — Quais são os principais problemas relatados pelos clientes sobre entrega?

IDs recuperados: ['95e7f613e3dfe80a79306a8c80084739', '65e3d04aef73660b0b2145ef3d5762cd', 'a83253201bb3ee4c020d9b42f1cfac38', 'e8d1bc0188ba7974e847024cab472cdb', 'cbc6f7cd85fd98b26a724ccb2e9c74b2']

RESPOSTA:
 Com base nas avaliações fornecidas, os principais problemas relatados pelos clientes sobre a entrega são:

* **Demora excessiva e atrasos:** Os clientes relatam muita demora na entrega [Review ID: 95e7f613e3dfe80a79306a8c80084739] e atrasos (mencionado mesmo em avaliação com nota 5) [Review ID: 65e3d04aef73660b0b2145ef3d5762cd], com registros de atrasos expressivos que chegam a 23 [Review ID: 95e7f613e3dfe80a79306a8c80084739] e 24 dias [Review ID: cbc6f7cd85fd98b26a724ccb2e9c74b2].
* **Não entrega dos pedidos / Produtos que não chegam:** Há relatos diretos de que o pedido ou produto não foi entregue [Review ID: a83253201bb3ee4c020d9b42f1cfac38, e8d1bc0188ba7974e847024cab472cdb, cbc6f7cd85fd98b26a724

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Q02 — O que os consumidores mais elogiam na experiência de compra?

IDs recuperados: ['029f6693f6fa41f1acd19f729e4e8b39', 'bf17681b8e58780d5064140aafa8f74d', '87bf7ea0b6a55642abc6b4dae8364d16', '7b89e676634561673670ec8b54ae202a', 'e141ad15361f70095fda499096a56ffc']

RESPOSTA:
 Com base nas avaliações fornecidas, os consumidores elogiam os seguintes aspectos da experiência de compra:

* **Qualidade geral da loja e do serviço:** Os clientes expressam satisfação geral com a loja, fazendo elogios e recomendações para amigos e familiares [Review ID: 029f6693f6fa41f1acd19f729e4e8b39, Review ID: bf17681b8e58780d5064140aafa8f74d, Review ID: 87bf7ea0b6a55642abc6b4dae8364d16].
* **Atendimento:** O atendimento é citado explicitamente como ótimo [Review ID: 87bf7ea0b6a55642abc6b4dae8364d16].
* **Agilidade na entrega:** A rapidez e a qualidade da entrega são destacadas como pontos positivos [Review ID: 87bf7ea0b6a55642abc6b4dae8364d16, Review ID: 7b89e676634561673670ec8b54ae202a].
* **Qualidade do

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Q03 — Quais problemas aparecem nas avaliações de clientes insatisfeitos?

IDs recuperados: ['87fb462b15995bf3c5b6fba9d9678e04', 'cb30c1d49c50132378faa2a855dbbb6d', '8d0f11dad4a5706b1a970995c6585605', 'd95962b1b4c23be31196b750630d00d6', 'b2fa7acac4cbfa62f9991421ff55545f']

RESPOSTA:
 Com base nas avaliações fornecidas, os clientes insatisfeitos relataram os seguintes problemas:

* **Insatisfação com o produto:** O produto não correspondeu às expectativas [Review ID: 87fb462b15995bf3c5b6fba9d9678e04] e foi classificado com "péssima qualidade" [Review ID: 8d0f11dad4a5706b1a970995c6585605].
* **Problemas de comunicação e rastreamento:** Houve falta de comunicação da empresa com o cliente e ausência de atualização de status do pedido [Review ID: cb30c1d49c50132378faa2a855dbbb6d].
* **Atrasos e não entrega:** Clientes relataram que o prometido não foi cumprido [Review ID: b2fa7acac4cbfa62f9991421ff55545f] e que não receberam o produto mesmo após já terem pago a primeira parcela [Review ID: 

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Q04 — Quais transtornos são relatados em pedidos com atraso?

IDs recuperados: ['95e7f613e3dfe80a79306a8c80084739', 'b7fe825a931a313b8aa2d18e8004b328', 'd5b353235f98e304b065c6b74c57fd3d', 'cf6506ccf9e5680269e733ff320d514d', '6b50a1c86b75347d9780edc40d28d5a5']

RESPOSTA:
 Com base nas avaliações fornecidas, os clientes relatam os seguintes transtornos associados a pedidos com atraso:

* **Não recebimento do produto:** O transtorno principal e comum a todas as avaliações citadas é a falta de entrega do pedido dentro do prazo ou a ausência total de recebimento [Review ID: 95e7f613e3dfe80a79306a8c80084739, Review ID: b7fe825a931a313b8aa2d18e8004b328, Review ID: d5b353235f98e304b065c6b74c57fd3d, Review ID: cf6506ccf9e5680269e733ff320d514d, Review ID: 6b50a1c86b75347d9780edc40d28d5a5].
* **Prejuízo financeiro e impacto profissional:** Uma cliente relata estar sofrendo prejuízo financeiro porque é nutricionista e depende do produto para conseguir atender à sua clientela [Review ID: cf6506ccf

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3935: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Q05 — Qual será a previsão do tempo amanhã em São Paulo?

IDs recuperados: ['20b09f99fa6ba25ae603fe6946064eb6', '30ca86c00c0eedaf6ea08a6261404cc2', '3ab3ec3a76b46d8fd46f7da4f30d4240', '254d7f4c6c49493b571abdd4840b70b9', '92bbc6038b05def6cd65df5a404f947c']

RESPOSTA:
 Não foram encontradas avaliações suficientes na base de conhecimento sobre este assunto.

Evidência 1 | Review ID: 20b09f99fa6ba25ae603fe6946064eb6 | Nota: 3 | Atraso: False
Não posso avaliar ainda não chegou.

Evidência 2 | Review ID: 30ca86c00c0eedaf6ea08a6261404cc2 | Nota: 1 | Atraso: False
Ainda não recomendaria pq não recebi o produto

Evidência 3 | Review ID: 3ab3ec3a76b46d8fd46f7da4f30d4240 | Nota: 5 | Atraso: False
entrega antes do prazo previsto, muito bom.

Evidência 4 | Review ID: 254d7f4c6c49493b571abdd4840b70b9 | Nota: 5 | Atraso: False
Fiz o pedido em uma quarta feira e recebi na sexta. Recomendo.

Evidência 5 | Review ID: 92bbc6038b05def6cd65df5a404f947c | Nota: 3 | Atraso: False
Aguardando a entrega. Ainda

## 5. Métricas de recuperação: Precision@5, Recall@5 e MRR

As métricas abaixo só são calculadas quando `review_ids_relevantes` foi preenchido com uma referência humana.

- **Precision@5:** proporção dos cinco resultados recuperados que são relevantes.
- **Recall@5:** proporção das avaliações relevantes de referência que apareceram entre os cinco resultados.
- **MRR:** posição do primeiro resultado relevante; quanto mais cedo ele aparece, maior o MRR.

A precisão usa K=5 como denominador mesmo se o Chroma retornar menos de cinco documentos, para manter a regra fixa de avaliação. Se preferir a precisão entre os documentos efetivamente retornados, registre essa escolha no relatório.

In [5]:
def metricas_por_pergunta(ids_relevantes, ids_recuperados, k=5):
    relevantes = set(map(str, ids_relevantes))
    recuperados = list(map(str, ids_recuperados[:k]))

    if not relevantes:
        return {
            "precision_at_5": np.nan,
            "recall_at_5": np.nan,
            "mrr": np.nan
        }

    acertos = sum(1 for rid in recuperados if rid in relevantes)
    precision = acertos / k
    recall = acertos / len(relevantes)

    reciprocal_rank = 0.0
    for posicao, rid in enumerate(recuperados, start=1):
        if rid in relevantes:
            reciprocal_rank = 1 / posicao
            break

    return {
        "precision_at_5": precision,
        "recall_at_5": recall,
        "mrr": reciprocal_rank
    }

linhas_metricas = []
for r in resultados_avaliacao:
    m = metricas_por_pergunta(
        r["review_ids_relevantes"],
        r["ids_recuperados"],
        k=5
    )
    linhas_metricas.append({
        "id_pergunta": r["id_pergunta"],
        "pergunta": r["pergunta"],
        "qtd_referencias": len(r["review_ids_relevantes"]),
        **m
    })

df_metricas_retrieval = pd.DataFrame(linhas_metricas)
display(df_metricas_retrieval)

print("\nMédias (somente perguntas com IDs de referência preenchidos):")
display(
    df_metricas_retrieval[
        ["precision_at_5", "recall_at_5", "mrr"]
    ].mean().to_frame("media")
)

,id_pergunta,pergunta,qtd_referencias,precision_at_5,recall_at_5,mrr
0,Q01,Quais são os principais problemas relatados pe...,4,0.8,1.000000,1.0
1,Q02,O que os consumidores mais elogiam na experiên...,7,0.8,1.000000,1.0
2,Q03,Quais problemas aparecem nas avaliações de cli...,4,0.6,1.000000,1.0
3,Q04,Quais transtornos são relatados em pedidos com...,3,0.4,0.666667,0.5
4,Q05,Qual será a previsão do tempo amanhã em São Pa...,0,NaN,NaN,NaN



Médias (somente perguntas com IDs de referência preenchidos):


,media
precision_at_5,0.650000
recall_at_5,0.916667
mrr,0.875000


In [6]:

for r in resultados_avaliacao:

    relevantes = set(r["review_ids_relevantes"])
    recuperados = r["ids_recuperados"][:5]

    acertos = len(set(recuperados) & relevantes)

    print(f"\n{r['id_pergunta']}")
    print("Referências:", len(relevantes))
    print("Recuperados:", len(recuperados))
    print("Acertos:", acertos)

    if relevantes:
        print("Precision:", acertos / 5)
        print("Recall:", acertos / len(relevantes))



Q01
Referências: 4
Recuperados: 5
Acertos: 4
Precision: 0.8
Recall: 1.0

Q02
Referências: 4
Recuperados: 5
Acertos: 4
Precision: 0.8
Recall: 1.0

Q03
Referências: 3
Recuperados: 5
Acertos: 3
Precision: 0.6
Recall: 1.0

Q04
Referências: 3
Recuperados: 5
Acertos: 2
Precision: 0.4
Recall: 0.6666666666666666

Q05
Referências: 0
Recuperados: 5
Acertos: 0


## 6. Avaliar fidelidade, relevância e abstenção

Essas avaliações precisam de uma referência humana ou de um avaliador automatizado validado. Para uma entrega acadêmica pequena, recomendo dois avaliadores atribuindo notas de 1 a 5:

- **Fidelidade (1–5):** todas as afirmações da resposta são sustentadas pelas evidências citadas?
- **Relevância (1–5):** a resposta responde diretamente à pergunta?
- **Abstenção correta:** quando a pergunta não tem evidência suficiente, o sistema se abstém? E quando há evidência, ele evita uma recusa indevida?

Não atribua nota automaticamente só porque o Gemini gerou uma resposta. Leia a resposta e compare com os comentários recuperados.

In [8]:
linhas_julgamento = []

for r in resultados_avaliacao:

    resposta_lower = r["resposta"].lower()

    abstencao_detectada = (
        "não foram encontradas avaliações suficientes"
        in resposta_lower
    )

    linhas_julgamento.append({
        "id_pergunta": r["id_pergunta"],
        "pergunta": r["pergunta"],
        "espera_abstencao": r["espera_abstencao"],
        "abstencao_detectada": abstencao_detectada,
        "abstencao_correta": (
            abstencao_detectada == r["espera_abstencao"]
        ),
        "faithfulness_1a5": None,
        "answer_relevancy_1a5": None,
        "observacoes": ""
    })

df_julgamento = pd.DataFrame(linhas_julgamento)

# Avaliação manual das respostas
# Notas de 1 a 5

notas = {
    "Q01": {"faithfulness": 5, "relevancy": 5, "observacao": "Resposta fundamentada nas avaliações recuperadas."},
    "Q02": {"faithfulness": 5, "relevancy": 5, "observacao": "Identificou corretamente os elogios dos consumidores."},
    "Q03": {"faithfulness": 5, "relevancy": 4, "observacao": "Resposta fundamentada, mas poderia ser mais completa."},
    "Q04": {"faithfulness": 5, "relevancy": 5, "observacao": "Resposta coerente com os relatos de atraso."},
    "Q05": {"faithfulness": None, "relevancy": None, "observacao": "Não deveria exibir nenhuma review, pois a pergunta é fora de escopo."},
}

for i, linha in df_julgamento.iterrows():

    pergunta_id = linha["id_pergunta"]

    if pergunta_id in notas:
        df_julgamento.loc[i, "faithfulness_1a5"] = (
            notas[pergunta_id]["faithfulness"]
        )

        df_julgamento.loc[i, "answer_relevancy_1a5"] = (
            notas[pergunta_id]["relevancy"]
        )

        df_julgamento.loc[i, "observacoes"] = (
                    notas[pergunta_id]["observacao"]
                )

display(df_julgamento)


,id_pergunta,pergunta,espera_abstencao,abstencao_detectada,abstencao_correta,faithfulness_1a5,answer_relevancy_1a5,observacoes
0,Q01,Quais são os principais problemas relatados pe...,False,False,True,5,5,Resposta fundamentada nas avaliações recuperadas.
1,Q02,O que os consumidores mais elogiam na experiên...,False,False,True,5,5,Identificou corretamente os elogios dos consum...
2,Q03,Quais problemas aparecem nas avaliações de cli...,False,False,True,5,4,"Resposta fundamentada, mas poderia ser mais co..."
3,Q04,Quais transtornos são relatados em pedidos com...,False,False,True,5,5,Resposta coerente com os relatos de atraso.
4,Q05,Qual será a previsão do tempo amanhã em São Pa...,True,True,True,None,None,"Não deveria exibir nenhuma review, pois a perg..."


## 7. Consolidar e exportar os resultados

Preencha as notas de fidelidade e relevância após a revisão humana. A taxa de abstenção correta é calculada sobre todos os casos classificados como respondíveis ou não respondíveis; revise também falsos positivos e falsas recusas individualmente.

In [9]:
print("RESULTADOS DA AVALIAÇÃO RAG")

print("\nRecuperação:")
display(df_metricas_retrieval[
    ["precision_at_5", "recall_at_5", "mrr"]
].mean())

print("\nGeração:")
print(
    "Faithfulness:",
    df_julgamento["faithfulness_1a5"].mean()
)

print(
    "Answer Relevancy:",
    df_julgamento["answer_relevancy_1a5"].mean()
)

print("\nAbstenção:")
print(
    "Taxa de acerto:",
    df_julgamento["abstencao_correta"].mean()
)

RESULTADOS DA AVALIAÇÃO RAG

Recuperação:


precision_at_5    0.650000
recall_at_5       0.916667
mrr               0.875000
dtype: float64


Geração:
Faithfulness: 5.0
Answer Relevancy: 4.75

Abstenção:
Taxa de acerto: 1.0
